# Übung 3: Augmentation, Regularisierung und XAI

## Worum geht es in dieser Übung?

Ein Bildklassifikationsmodell soll das **Objekt im Bild** erkennen, zum Beispiel einen Vogel oder ein Auto. Es kann aber auch eine bequeme, zufällige Abkürzung lernen: etwa ein Logo in einer Ecke, einen bestimmten Hintergrund oder eine Bildfarbe. Solche zufällig mit dem Label zusammenhängenden Hinweise heißen *spurious correlations* (Scheinkorrelationen).

Die Übung ist als zusammenhängende Untersuchung aufgebaut:

1. Sie bauen eine kleine, kontrollierbare Ausgangssituation.
2. Sie beobachten, wie stark ein Modell ohne Gegenmaßnahmen auswendig lernt.
3. Sie testen, ob Augmentation und Dropout die Generalisierung verbessern.
4. Sie erzeugen absichtlich eine Scheinkorrelation und zeigen, dass eine gute Accuracy allein nicht genügt.
5. Sie prüfen mit XAI, **welche Bildbereiche** die Entscheidung beeinflussen.
6. Sie leiten daraus sinnvolle Maßnahmen für echte Projekte ab.

**Leitfrage:** Lernt das Modell das Motiv - oder nur einen Hinweis, der im Trainingsdatensatz zufällig praktisch ist?

**Wichtig:** Eine hohe Test-Accuracy ist nur dann aussagekräftig, wenn der Test die reale Anwendungssituation abbildet. Deshalb enthält Aufgabe 4 absichtlich einen zweiten, anspruchsvolleren Test.


## Aufgabe 1 - CIFAR-10-Subset vorbereiten

Wählen Sie zwei bis vier CIFAR-10-Klassen, zum Beispiel cat, dog, automobile und truck. Erstellen Sie ein kleines Trainingsset und ein davon getrenntes Testset.

**Warum ein Subset?** Mit wenigen Klassen und bewusst wenig Trainingsbildern geht das Training schnell und Overfitting wird leichter sichtbar. Das ist hier erwünscht: Erst wenn Sie den Effekt sehen, können Sie beurteilen, ob die Gegenmaßnahmen helfen.

**Was muss getrennt bleiben?**

- Trainingsbilder: Daraus lernt das Modell Gewichte.
- Validierungsbilder (empfohlen): Damit vergleichen Sie Modellvarianten und wählen Einstellungen.
- Testbilder: Diese bleiben bis zur abschließenden Bewertung unangetastet.

Halten Sie Klassen, Anzahl der Bilder, Zufalls-Seed und Datenaufteilung für alle folgenden Experimente gleich. Sonst könnte ein Unterschied in den Lernkurven von anderen Daten statt von Augmentation oder Dropout kommen.

**Dokumentieren Sie:** Welche Klassen wurden gewählt? Wie viele Bilder liegen in Train/Valid/Test? Sehen die Klassen für Menschen plausibel unterscheidbar aus?


## Aufgabe 2 - Ohne Augmentation trainieren

Trainieren Sie ein kleines CNN ohne Augmentation und plotten Sie Train- und Validierungs-Accuracy pro Epoche. Den Test nutzen Sie erst, nachdem Sie Ihre Modellentscheidung getroffen haben.

**Sinn dieses Laufs:** Das ist Ihre Baseline. Das Modell sieht jedes Trainingsbild immer exakt gleich. Es kann daher einzelne Positionen, Ausschnitte oder Hintergründe sehr leicht auswendig lernen. Ohne diesen Vergleichslauf lässt sich später nicht beurteilen, ob die Regularisierung wirklich etwas verbessert.

Achten Sie auf die Lücke zwischen den Kurven:

- Train hoch, Valid deutlich niedriger: typisches Overfitting.
- Beide niedrig: Modell, Lernrate oder Trainingsdauer passen vermutlich noch nicht.
- Beide hoch und nah beieinander: für dieses Subset generalisiert das Modell bereits gut.

**Fairer Vergleich:** Behalten Sie Architektur, Optimizer, Lernrate, Epochen und Datenaufteilung in Aufgabe 3 bei. Verändern Sie dort nur Augmentation und Dropout.


## Aufgabe 3 - Mit Augmentation und Dropout trainieren

Trainieren Sie dieselbe Architektur mit RandomCrop, RandomHorizontalFlip, einem vorsichtigen ColorJitter und Dropout. Vergleichen Sie die Lernkurven mit Aufgabe 2.

**Was verändert Augmentation?** Sie erzeugt bei jedem Training leicht veränderte, aber fachlich noch korrekte Varianten eines Bildes. Das Modell soll dadurch lernen: "Ein Objekt bleibt dieselbe Klasse, auch wenn es etwas verschoben, gespiegelt oder anders beleuchtet ist." Die Augmentation gehört nur in den Trainings-Dataloader; Validierungs- und Testbilder müssen unverändert bleiben.

**Was verändert Dropout?** Beim Training werden zufällig einige Aktivierungen auf null gesetzt. Das Netz kann sich dadurch weniger auf einzelne Neuronen oder einzelne Bildhinweise verlassen. Beim Auswerten ist Dropout ausgeschaltet (model.eval()).

**Erwartung:** Die Trainings-Accuracy darf anfangs niedriger sein als in Aufgabe 2, weil die Aufgabe absichtlich schwieriger wird. Entscheidend ist nicht die höchste Trainings-Accuracy, sondern eine kleinere Lücke zur Validierung und eine bessere oder stabilere Validierungs-Accuracy.

**Reflexion:** Welche Transformationen sind für Ihre Klassen plausibel? Horizontal spiegeln ist bei vielen CIFAR-Objekten sinnvoll; eine starke Farbänderung kann dagegen die Klasse verfälschen.


## Aufgabe 4 - Spurious Artefact erzeugen

Fügen Sie bei **einer** Klasse im Training ein kleines, auffälliges Quadrat oder Logo in eine feste Ecke ein; zum Beispiel nur bei allen Bildern der Klasse cat. Das Objekt selbst darf dadurch nicht verdeckt werden.

**Warum machen wir absichtlich einen Fehler?** So wird sichtbar, dass das Modell nicht weiß, was inhaltlich wichtig ist. Es minimiert nur den Fehler auf den Trainingsdaten. Wenn ein Logo dort perfekt zum Label passt, ist das Logo für das Modell ein sehr bequemer Prädiktor - auch wenn es in der echten Aufgabe nichts mit Katzen zu tun hat.

Bewerten Sie zwei Testbedingungen:

| Testbedingung | Vorbereitung | Erwartete Aussage |
| --- | --- | --- |
| **Bias bleibt erhalten** | Die Bilder der manipulierten Klasse tragen weiter das Logo. | Die Accuracy kann gut aussehen, obwohl das Modell das falsche Merkmal nutzt. |
| **Bias ist gebrochen** | Entfernen Sie das Logo bei dieser Klasse oder setzen Sie es auch bei anderen Klassen ein. | Fällt die Accuracy besonders für die manipulierte Klasse, war das Logo wichtiger als der Bildinhalt. |

Vergleichen Sie nicht nur die Gesamt-Accuracy, sondern auch die Accuracy pro Klasse und eine Confusion Matrix. Eine gute Gesamtzahl kann ein ernstes Problem in einer einzelnen Klasse verdecken.

**Analogie:** Wenn auf allen Trainingsfotos von Hunden zufällig ein Tierheim-Logo steht, kann ein Modell Logo = Hund lernen. Im Test mit demselben Logo wirkt es gut; auf echten Fotos ohne Logo versagt es.


## Aufgabe 5 - XAI prüfen

Erstellen Sie Saliency- oder Occlusion-Maps für mehrere Bilder und bewerten Sie, ob das Modell auf das Objekt oder auf das Artefakt achtet.

**Was beantwortet XAI hier?** XAI erklärt nicht, ob die Vorhersage richtig ist, sondern welche Pixel die Vorhersage am stärksten beeinflusst haben. Genau das brauchen wir nach Aufgabe 4: Die Accuracy allein verrät nicht, ob das Modell sinnvoll entschieden hat.

- **Saliency Map:** Berechnen Sie den Gradienten des vorhergesagten Klassen-Scores nach den Eingabepixeln. Große Werte bedeuten: Eine kleine Änderung an diesem Pixel würde den Score stark verändern.
- **Occlusion Map:** Verdecken Sie nacheinander kleine Bildbereiche und messen Sie, wie stark der Score der Zielklasse sinkt. Ein starker Einbruch zeigt einen wichtigen Bereich. Diese Methode ist langsamer, aber oft leichter zu erklären.

Legen Sie die Karte transparent über das Originalbild. Liegt der warme bzw. wichtige Bereich wiederholt auf der Logo-Ecke, ist das ein konkreter Hinweis auf die Scheinkorrelation. Liegt er auf dem Tier/Fahrzeug, spricht das eher für eine inhaltliche Entscheidung.

Betrachten Sie mehrere korrekte und falsche Beispiele - nicht nur ein besonders passendes Bild. Das Mitteln mehrerer normierter Maps kann ein Muster sichtbar machen, ersetzt aber nicht die Einzelbilder.


## Aufgabe 6 - Maßnahmen

Beschreiben Sie drei konkrete Maßnahmen gegen das gefundene Overfitting auf Artefakte. Begründen Sie jeweils, **welchen Teil des Problems** die Maßnahme adressiert.

Mögliche Ansatzpunkte:

1. **Daten bereinigen oder ausbalancieren:** Logo/Wasserzeichen entfernen oder bewusst gleichmäßig über alle Klassen verteilen. Das beseitigt die falsche Korrelation direkt in den Daten.
2. **Gezielte Augmentation:** Logos ausschneiden, verdecken (Random Erasing/Cutout) oder ihre Position variieren. Das macht es schwieriger, eine feste Ecke als Abkürzung zu verwenden. Allgemeine Augmentation allein garantiert das aber nicht.
3. **Realistische Gegenvalidierung:** Einen zusätzlichen Testsplit ohne Logo bzw. aus einer anderen Quelle anlegen und Metriken pro Gruppe berichten. Damit fällt das Problem vor dem Einsatz auf.
4. **Fachliche Datenprüfung und XAI-Stichproben:** Vor dem Training auf wiederkehrende Hintergründe, Quellen und Wasserzeichen prüfen; nach dem Training Erklärkarten stichprobenartig kontrollieren.

**Abschluss:** Formulieren Sie in zwei bis drei Sätzen Ihr Ergebnis: Hat Augmentation/Dropout die Generalisierung verbessert? Was passierte, als der Bias brach? Welche Evidenz liefern Ihre XAI-Karten dafür, worauf das Modell tatsächlich achtet?
